# Calibration

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")
sys.path.append("../prediction_src")

import pandas as pd

import numpy as np

from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm

import random
import pickle
import sklearn

from collections import defaultdict

from sklearn.model_selection import KFold, cross_validate 

from evaluate_prediction_model import *

from sklearn.base import BaseEstimator, ClassifierMixin
import statsmodels.api as sm

from fit_prediction_model import *
from calibration import *

from prediction_pipeline import model_pipeline

import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, roc_auc_score,brier_score_loss,roc_curve, confusion_matrix

from sklearn.metrics import classification_report

from params import MET_LIST, CALIBRATION_MET_LIST

import copy

In [3]:
with open("../models/DAG_model_durchfall.pkl","rb") as f:
    basis_model_dict = pickle.load(f)

In [4]:
with open("../models/DAG_model_durchfall.pkl","rb") as f:
    ppv_model_dict = pickle.load(f)

In [5]:
with open("../models/DAG_model_durchfall.pkl","rb") as f:
    fpr_model_dict = pickle.load(f)

In [6]:
with open("../models/DAG_model_durchfall.pkl","rb") as f:
    youden_model_dict = pickle.load(f)

In [44]:
with open("../models/mens_data_model.pkl","rb") as f:
    mens_model_dict = pickle.load(f)

In [7]:
basis_model_dict[list(basis_model_dict)[0]]["model"].threshold

0.5

In [ ]:


calc_eval_metrics(basis_model_dict, CALIBRATION_MET_LIST).round(3)

,Constrained FPR Cutoff,Max PPV Cutoff,Youden Cutoff
Model,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.05,0.16,0.02


In [10]:
fpr_cutoff, metrics_for_fpr = compute_and_save_cutoff(fpr_model_dict,"FPR")

FPR optimal cutoff: 0.047

Metrics for this cutoff:


,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.182,0.951,0.074,0.049,0.182,0.935,76,958,342,18598


In [11]:
ppv_cutoff, metrics_for_ppv = compute_and_save_cutoff(ppv_model_dict,"PPV")

PPV optimal cutoff: 0.156

Metrics for this cutoff:


,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.002,0.999,0.091,0.001,0.002,0.979,1,10,417,19546


In [12]:
basis_model_dict[list(basis_model_dict)[0]]["model"].threshold

0.5

In [13]:
youden_cutoff, metrics_for_youden = compute_and_save_cutoff(youden_model_dict,"youden")

youden optimal cutoff: 0.02

Metrics for this cutoff:


,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.517,0.778,0.047,0.222,0.517,0.773,216,4337,202,15219


In [45]:
youden_cutoff

0.01954109442520162

In [47]:
mens_youden_cutoff, metrics_for_mens_youden =compute_and_save_cutoff(mens_model_dict,"youden")

youden optimal cutoff: 0.049

Metrics for this cutoff:


,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.513,0.679,0.075,0.321,0.513,0.671,61,749,58,1585


In [14]:
basis_model_dict[list(basis_model_dict)[0]]["model"].threshold

0.5

In [15]:
default_cutoff, metrics_for_default = compute_and_save_cutoff(basis_model_dict,"default")

default optimal cutoff: 0.5

Metrics for this cutoff:


,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.0,1.0,NaN,0.0,0.0,0.979,0,0,418,19556


In [16]:
basis_model_dict[list(basis_model_dict)[0]]["model"].threshold

0.5

In [18]:
ppv_cutoff

0.156312625250501

In [41]:
youden_model_dict[list(youden_model_dict.keys())[0]]["y_test_true"].mean()

0.02092720536697707

In [19]:
basis_model_dict[list(basis_model_dict)[0]]["model"].threshold

0.5

In [20]:
ppv_model_dict=update_with_new_cutoff(basis_model_dict,ppv_cutoff)

In [21]:
basis_model_dict[list(basis_model_dict)[0]]["model"].threshold

0.5

In [22]:
fpr_model_dict=update_with_new_cutoff(basis_model_dict,fpr_cutoff)

In [23]:
youden_model_dict=update_with_new_cutoff(basis_model_dict,youden_cutoff)

In [24]:
ppv_model_dict[list(ppv_model_dict.keys())[0]]["model"].threshold

0.156312625250501

In [25]:
basis_model_dict[list(basis_model_dict.keys())[0]]["model"].threshold

0.5

In [26]:
fpr_model_dict[list(fpr_model_dict.keys())[0]]["model"].threshold

0.04731142174755568

In [27]:
calc_eval_metrics(basis_model_dict, MET_LIST)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.66,0.02,0.26,1.07


In [28]:
calc_eval_metrics(ppv_model_dict, MET_LIST)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.66,0.02,0.26,1.07


In [29]:
calc_eval_metrics(fpr_model_dict, MET_LIST)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.66,0.02,0.26,1.07


In [30]:
calc_eval_metrics(youden_model_dict, MET_LIST)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.66,0.02,0.26,1.07


In [32]:
metrics_for_fpr["method"] = "FPR_optimum"
metrics_for_ppv["method"] = "PPV_optimum"
metrics_for_youden["method"] = "Youden_optimum"
metrics_for_default["method"] = "Default_cutoff"

In [39]:
pd.concat([metrics_for_default,metrics_for_fpr,metrics_for_ppv,metrics_for_youden]).round(3)

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN,method
0,0.000,1.000,NaN,0.000,0.000,0.979,0,0,418,19556,Default_cutoff
0,0.182,0.951,0.074,0.049,0.182,0.935,76,958,342,18598,FPR_optimum
0,0.002,0.999,0.091,0.001,0.002,0.979,1,10,417,19546,PPV_optimum
0,0.517,0.778,0.047,0.222,0.517,0.773,216,4337,202,15219,Youden_optimum


In [35]:
basis_model_dict[list(basis_model_dict.keys())[0]]["model"].coef_table_.round(3)

,feature,coef,odds_ratio,p_value,ci_lower,ci_upper
0,Intercept,-4.182,0.015,0.000,-4.251,-4.114
1,Menstruationsschmerzen,0.809,2.246,0.000,0.681,0.937
2,Bauchschmerzen,0.818,2.267,0.000,0.688,0.949
3,Verdauungsstörungen,0.378,1.460,0.041,0.016,0.741
4,Verstopfung,-0.112,0.894,0.829,-1.125,0.902
5,Blutung,0.476,1.610,0.000,0.249,0.703
6,Urinieren,0.813,2.255,0.000,0.465,1.161
7,Dyspareunie,0.361,1.435,0.625,-1.087,1.809


In [ ]:
with open("../models//DAG_model_youden_final.pkl","wb") as f:
    pickle.dump(youden_model_dict,f)

In [37]:
with open("../models/DAG_model_ppv_durchfall.pkl","wb") as f:
    pickle.dump(ppv_model_dict,f)

In [38]:
with open("../models/DAG_model_fpr_durchfall.pkl","wb") as f:
    pickle.dump(fpr_model_dict,f)